# F1 Driver Head-to-Head Comparison

Compare two drivers' careers: race wins, points over time, and qualifying/race head-to-head record when they were teammates or contemporaries. Change `DRIVER_A` / `DRIVER_B` below to compare anyone in the dataset.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "races.csv" in files:
        DATA_DIR = root
        break
print("Using data dir:", DATA_DIR)

races = pd.read_csv(f"{DATA_DIR}/races.csv")
results = pd.read_csv(f"{DATA_DIR}/results.csv")
drivers = pd.read_csv(f"{DATA_DIR}/drivers.csv")

results = results.merge(races[["raceId", "year", "date", "name"]], on="raceId")
results = results.merge(drivers[["driverId", "forename", "surname", "driverRef"]], on="driverId")
results["driverName"] = results["forename"] + " " + results["surname"]

In [ ]:
DRIVER_A = "max_verstappen"
DRIVER_B = "lewis_hamilton"

a = results[results.driverRef == DRIVER_A]
b = results[results.driverRef == DRIVER_B]

def summarize(df, ref):
    wins = (df.positionOrder == 1).sum()
    podiums = (df.positionOrder <= 3).sum()
    races_entered = len(df)
    points = df.points.sum()
    print(f"{ref}: {races_entered} races, {wins} wins, {podiums} podiums, {points:.0f} career points")

summarize(a, DRIVER_A)
summarize(b, DRIVER_B)

## Career points progression

In [ ]:
a_by_year = a.groupby("year").points.sum().cumsum()
b_by_year = b.groupby("year").points.sum().cumsum()

plt.figure(figsize=(10, 5))
plt.plot(a_by_year.index, a_by_year.values, label=DRIVER_A, marker="o")
plt.plot(b_by_year.index, b_by_year.values, label=DRIVER_B, marker="o")
plt.xlabel("Season")
plt.ylabel("Cumulative career points")
plt.title(f"{DRIVER_A} vs {DRIVER_B}: cumulative points by season")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Head-to-head when racing in the same event

In [ ]:
shared_races = set(a.raceId) & set(b.raceId)
print(f"Races both drivers entered: {len(shared_races)}")

a_wins_h2h = 0
b_wins_h2h = 0
for race_id in shared_races:
    a_pos = a[a.raceId == race_id].positionOrder.iloc[0]
    b_pos = b[b.raceId == race_id].positionOrder.iloc[0]
    if a_pos < b_pos:
        a_wins_h2h += 1
    elif b_pos < a_pos:
        b_wins_h2h += 1

print(f"{DRIVER_A} finished ahead: {a_wins_h2h} times")
print(f"{DRIVER_B} finished ahead: {b_wins_h2h} times")

## Wins per season

In [ ]:
a_wins_year = a[a.positionOrder == 1].groupby("year").size()
b_wins_year = b[b.positionOrder == 1].groupby("year").size()

years = sorted(set(a_wins_year.index) | set(b_wins_year.index))
a_vals = [a_wins_year.get(y, 0) for y in years]
b_vals = [b_wins_year.get(y, 0) for y in years]

import numpy as np
x = np.arange(len(years))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar(x - width/2, a_vals, width, label=DRIVER_A)
plt.bar(x + width/2, b_vals, width, label=DRIVER_B)
plt.xticks(x, years, rotation=45)
plt.ylabel("Wins")
plt.title(f"{DRIVER_A} vs {DRIVER_B}: wins per season")
plt.legend()
plt.tight_layout()
plt.show()